# Laboratorium 1: Dekoratory, Deskryptory i Generatory
### Skoroszyt główny

---

## Cele Laboratorium
Celem dzisiejszych zajęć jest opanowanie zaawansowanych konstrukcji języka Python, które są niezbędne do projektowania nowoczesnej architektury aplikacji.

### System Wspomagania AI (Tutor)
W trakcie rozwiązywania zadań możesz korzystać z pomocy dedykowanego tutora AI. System oferuje 6 poziomów wsparcia:
1. **Ogólna wskazówka**: Sugestia kierunku rozwiązania.
2. **Pseudokod**: Logiczny opis algorytmu.
3. **Mały fragment kodu**: Kluczowa linia lub konstrukcja.
4. **Częściowa implementacja**: Szkielet kodu do uzupełnienia.
5. **Szczegółowe wyjaśnienie**: Analiza mechanizmu działania.
6. **Pełne rozwiązanie**: Dostępne w sytuacjach ostatecznych.

---

## 1. Dekoratory

### DEMO: Dekorator @timer 
Stwórz dekorator @timer, który będzie mierzył i wyświetlał czas wykonania funkcji.

In [ ]:
import time
import functools

def timer(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"Czas wykonania {func.__name__}: {end_time - start_time:.4f} s")
        return result
    return wrapper

@timer
def example_task():
    time.sleep(0.5)
    print("Zadanie zakończone.")

example_task()

Zadanie zakończone.
Czas wykonania example_task: 0.5051 s


### Zadanie 1: Liczba elementów listy
Stwórz dekorator, który będzie odpowiedzialny za wyświetlanie liczby elementów listy, jeśli jakakolwiek lista pojawi się w parametrach funkcji dekorowanej. 

**Protip:** użyj isinstance do sprawdzenia czy parametr jest listą. Pamiętaj o zachowaniu metadanych funkcji.

In [1]:
import functools

def show_list_length(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        for arg in args:
            if isinstance(arg, list):
                print(f'Liczba elementów {arg}: {len(arg)}')
                return func(*args, **kwargs)

        for key, value in kwargs.items():
            if isinstance(value, list):
                print(f'Liczba elementów {value}: {len(value)}')
        return func(*args, **kwargs)

    return wrapper

# Test:
@show_list_length
def process_data(data_list, name):
    print(f"Przetwarzanie {name}")

process_data([1, 2, 3, 4], 'Tomek')

Liczba elementów [1, 2, 3, 4]: 4
Przetwarzanie Tomek


### Zadanie 2: Logowanie do pliku
Stwórz dekorator, który będzie zapisywał w pliku *.log nazwę funkcji dekorowanej, datę oraz długość wykonania. Nazwa pliku będzie podana jako argument dekoratora.

**Protip:** użyj biblioteki datetime. Pamiętaj o tym, żeby dekoratory przyjęły metadanych funkcji dekorującej.

In [3]:
import functools
from datetime import datetime
import time

def logger(filename):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            with open(f'{filename}.log', 'a') as f:
                start = time.perf_counter()
                try:
                    return func(*args, **kwargs)
                finally:
                    end = time.perf_counter()
                    f.write(
                        f"\nFunction: {func.__name__}\n"
                        f"Date: {datetime.now()}\n"
                        f"Execution time: {end - start:.4f} s\n"
                    )
        return wrapper
    return decorator

@logger('logger_output')
def logger_test(message):
    print(message)

logger_test('This is an example for logger decorator')

This is an example for logger decorator


--- 
## 2. Deskryptory

### DEMO: Walidator e-mail klasy Student
Stwórz deskryptor, który będzie działał jako walidator email klasy Student. Klasa Student zawiera pola imie, nazwisko i email. Deskryptor ten powinien sprawdzać poprawność danych wprowadzanych podczas tworzenia lub modyfikowania instancji Student.

In [5]:
class EmailValidator:
    def __set_name__(self, owner, name):
        self.name = name

    def __set__(self, instance, value):
        if "@" not in value:
            raise ValueError(f"Błędny format adresu email: {value}")
        instance.__dict__[self.name] = value

class Student:
    email = EmailValidator()
    
    def __init__(self, imie, nazwisko, email):
        self.imie = imie
        self.nazwisko = nazwisko
        self.email = email

try:
    s = Student("Jan", "Kowalski", "jan.kowalski@wsei.edu.pl")
    print(f"Utworzono studenta: {s.email}")
    s.email = "invalid_at_email" # Powinno rzucić błąd
except ValueError as e:
    print(e)

Utworzono studenta: jan.kowalski@wsei.edu.pl
Błędny format adresu email: invalid_at_email


### Zadanie 3: Rejestrowanie dostępu
Stwórz klasę Uzytkownik. Klasa powinna zawierać atrybuty imie i wiek. Opracuj deskryptor, który będzie rejestrował dostęp do tych atrybutów za pomocą logowania. Deskryptor powinien logować informacje o odczycie (__get__) oraz zapisie (__set__) wartości atrybutu.

In [6]:
class AccessLogger:
    def __set_name__(self, owner, name):
        self.name = name
    
    def __get__(self, instance, owner):
        if instance is None:
            return self
        print(f'Returning value for {self.name}, class: {owner}, instance: {instance}')
        return instance.__dict__[self.name]
    
    def __set__(self, instance, value):
        print(f'Changing value of: {self.name} of instance {instance}')
        instance.__dict__[self.name] = value

class Uzytkownik:
    imie = AccessLogger()
    wiek = AccessLogger()

    def __init__(self, imie, wiek):
        self.imie = imie
        self.wiek = wiek

user = Uzytkownik('Tomek', 23)
print(user.imie)
print(user.wiek)

Changing value of: imie of instance <__main__.Uzytkownik object at 0x0000017C31BC37D0>
Changing value of: wiek of instance <__main__.Uzytkownik object at 0x0000017C31BC37D0>
Returning value for imie, class: <class '__main__.Uzytkownik'>, instance: <__main__.Uzytkownik object at 0x0000017C31BC37D0>
Tomek
Returning value for wiek, class: <class '__main__.Uzytkownik'>, instance: <__main__.Uzytkownik object at 0x0000017C31BC37D0>
23


--- 
## 3. Generatory i Iteratory

### DEMO: Generator Fibonacciego
Napisz klasę, która będzie implementowała generator ciągu Fibonacciego za pomocą metod magicznych __iter__() i __next__().

In [ ]:
class FibonacciGenerator:
    def __init__(self, limit):
        self.limit = limit
        self.a, self.b = 0, 1
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.limit:
            raise StopIteration
        
        result = self.a
        self.a, self.b = self.b, self.a + self.b
        self.count += 1
        return result

fib = FibonacciGenerator(10)
print(list(fib))

### Zadanie 4: Generator ciągu Collatza
Opracuj generator ciągu Collatza. Dla liczby naturalnej n, jeśli n jest parzyste, dziel przez 2; jeśli n jest nieparzyste, pomnóż przez 3 i dodaj 1, zaczynając od określonej liczby początkowej, aż do osiągnięcia wartości 1.

In [ ]:
def collatz_generator(n):
    if n < 1: 
        raise ValueError()
    
    yield n
    while n > 1:
        if n % 2 == 0:
            n = n // 2
        else:
            n = n * 3 + 1
        yield n

# Test:
for status in collatz_generator(10):
    print(f'{status:.0f}')

---

## Zadania do zrobienia w domu

Poniższe zadania stanowią rozszerzenie materiału i są przeznaczone dla osób chcących zgłębić temat zaawansowanych konstrukcji języka Python.

### Zadanie dodatkowe 1: Dekorator z autoryzacją
Stwórz dekorator `@require_role(role)`, który przyjmuje nazwę wymaganej roli jako argument. Dekorator powinien sprawdzać, czy w globalnym słowniku `current_user` klucz `role` jest zgodny z wymaganym. Jeśli nie, rzuć `PermissionError`.

In [ ]:
import functools

current_user = {"username": "admin", "role": "superuser"}

def require_role(role):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if current_user['role'] != role:
                raise PermissionError()
            
            return func(*args, **kwargs)
        return wrapper
    return decorator

@require_role('superuser')
def permission_for_superuser(value):
    print(f'user dict: {current_user}, value: {value}')

permission_for_superuser(20)

@require_role('guest')
def permission_for_guest(value):
    print(f'user dict: {current_user}, value: {value}')

permission_for_guest(10)



### Zadanie dodatkowe 2: Deskryptor z walidacją typu
Stwórz deskryptor `Typed`, który przyjmuje typ danych (np. `int`, `str`) w konstruktorze. Deskryptor powinien upewnić się, że zapisywana wartość jest tego typu. Jeśli nie, rzuć `TypeError`.

In [ ]:
class Typed:
    def __init__(self, type):
        self.type = type

    def __set_name__(self, owner, name):
        self.name = name
    
    def __set__(self, instance, value):
        if not isinstance(value, self.type):
            raise TypeError()
        
        instance.__dict__[self.name] = value

class User:
    album_no = Typed(int)

    def __init__(self, album_no):
        self.album_no = album_no

    def __str__(self):
        return f'This user album is {self.album_no}'

usr1 = User(14660)
print(usr1)

usr2 = User('tomek')
print(usr2)

### Zadanie dodatkowe 3: Nieskończony generator liczb pierwszych
Opracuj generator `prime_generator`, który zwraca kolejne liczby pierwsze. Następnie użyj wyrażenia generatorowego, aby stworzyć iterator zwracający tylko te liczby pierwsze, które kończą się cyfrą 7.

In [ ]:
def is_prime(n):
    if n <= 1: return False

    for i in range(2, int(n**0.5) + 1):
        if n%i == 0: return False
    
    return True

def prime_generator():
    i = 1
    while True:
        if is_prime(i): yield i
        i+=1

print('Prime numbers:')
prime_gen = prime_generator()
for i in range(10):
    print(next(prime_gen))

print('\n\n')

print('Ends with 7:\n')
prime_gen_endswith_7 = (x for x in prime_gen if x%10 == 7)
for i in range(10):
    print(next(prime_gen_endswith_7))